<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-04-classification/04_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup, W&B & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.datasets import load_breast_cancer
import wandb

# Inisialisasi W&B
run = wandb.init(project="breast-cancer-classification", name="logistic-regression-baseline")

# Load dataset langsung dari Scikit-Learn (Dijamin 100% selalu bisa diakses)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target # 0: Malignant (Ganas), 1: Benign (Jinak)

# Log raw data ke W&B Table
wandb.log({"raw_data": wandb.Table(dataframe=df.head(100))})
print("Dataset Shape:", df.shape)
df.head()

Cek Keseimbangan Kelas & Split Data

In [ ]:
# Cek proporsi target (Penting dalam klasifikasi!)
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target', hue='target', palette='Set2', legend=False)
plt.title("Distribusi Kelas (0: Ganas, 1: Jinak)")
plt.show()

# Pisahkan Fitur dan Target
X = df.drop(columns=['target'])
y = df['target']

# Split Data: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
wandb.config.update({"test_size": 0.2, "random_state": 42})

Training Model

In [ ]:
# Inisialisasi model Logistic Regression
# Parameter max_iter dinaikkan untuk mencegah ConvergenceWarning (Best Practice Profesional)
model = LogisticRegression(max_iter=10000)

# Proses Belajar
model.fit(X_train, y_train)
print("Training selesai!")

Prediksi & Evaluasi Metrik

In [ ]:
# Lakukan prediksi pada data Test
y_pred = model.predict(X_test)

# Hitung Metrik Evaluasi
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Log ke W&B
wandb.log({
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1_Score": f1
})

print("--- Hasil Evaluasi Model ---")
print(f"Accuracy : {acc:.4f} (Persentase tebakan benar secara keseluruhan)")
print(f"Precision: {prec:.4f} (Bila model menebak 'Jinak', seberapa akurat tebakannya?)")
print(f"Recall   : {rec:.4f} (Dari semua kasus 'Jinak' asli, berapa persen yang berhasil ditebak model?)")
print(f"F1-Score : {f1:.4f} (Nilai harmonis antara Precision dan Recall)")

Confusion Matrix Visualization

In [ ]:
# Memvisualisasikan di mana letak kesalahan tebakan model
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ditebak Ganas (0)', 'Ditebak Jinak (1)'],
            yticklabels=['Asli Ganas (0)', 'Asli Jinak (1)'])
plt.title('Confusion Matrix - Prediksi Kanker Payudara')
plt.ylabel('Kondisi Aktual')
plt.xlabel('Tebakan Model')
plt.show()

# Menampilkan Laporan Lengkap
print("\nClassification Report:\n", classification_report(y_test, y_pred))

wandb.finish()